# Prédiction de réachat produit

## Objectif

Prédire si un produit donné sera réacheté (reordered = 1) dans la prochaine commande d'un utilisateur

Ce modèle s'appuie sur les constats de l'analyse exploratoire:
* La fidélité se traduit par la récurrence, et non par le volume panier
* Les 10 premières commandes sont la fenêtre critique de fidélisation
* L'effet client domine l'effet catégorie dans le réachat

## Approche

1. **Feature engineering**: variable à 3 niveaux: utilisateur, produit, user x produit
2. **Split temporel**: dernière commande = test (on ne fait pas de split aléatoire, donc pas de data leakage)
3. **Comparaison**: Logistic Regression (baseline) vs Random Forest vs LightGBM
4. **Evaluation**: accuracy, precision, recall, F1, ROC-AUC

## Justification des outils

| Package | Rôle | Pourquoi ce choix |
|---|---|---|
| **scikit-learn** | Logistic Regression, Random Forest, métriques | Standard de l'industrie, API cohérente, reproductible |
| **LightGBM** | Gradient boosting | Plus rapide que XGBoost sur les gros datasets tabulaires, gère les catégorielles nativement |
| **seaborn / matplotlib** | Visualisation des résultats | Cohérence avec le reste du notebook, qualité publication |

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append("..")
from src.data_loader import get_data
from src.features import enrich_dataframe
from src.ml_features import build_all_features
from src.ml_pipeline import ReorderPipeline

# Style global
sns.set_theme(style="whitegrid", font_scale=1)
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.facecolor": "#fafafa",
    "figure.facecolor": "white",
})

COLORS = {
    "occasionnel": "#a8bfc1",
    "typique":     "#4290ba",
    "fidèle":      "#1b3a4b",
    "accent":      "#3cc494",
    "grey":        "#6b7280",
}
PAL_SEGMENTS = [COLORS["occasionnel"], COLORS["typique"], COLORS["fidèle"]]

In [2]:
# Chargement des données

data = get_data(
    raw_dir="../data/raw",
    processed_path="../data/processed/merged_orders.parquet"
)
data = enrich_dataframe(data)
print(f"Dataset: {data.shape[0]:,} lignes et {data.shape[1]} colonnes")

Chargement depuis le cache .parquet
Dataset: 32,434,489 lignes et 21 colonnes


## Etape 1. Split temporel

On sépare les données chronologiquement, et non aléatoirement:

* **Train**: toutes les commandes de chaque utilisateur sauf la dernière
* **Test**: la dernière commande de chaque utilisateur

C'est l'approche standard pour les problèmes de *next basket prediction*. Un split aléatoire causerait du **data leakage**. Le modèle verrait les commandes futures pour prédire le passé.

In [3]:
# Initialisation du pipeline
# sample_frac=0.3 pour itérer vite; passer à 1.0 pour le run final
pipeline = ReorderPipeline(data, sample_frac=0.3, random_state=42)
pipeline.split()

ETAPE 1 : Split temporel
  Train : 30,294,701 lignes
  Test  : 2,139,788 lignes


ReorderPipeline(data=          order_id  product_id  add_to_cart_order  reordered  user_id  \
0                2       33120                  1          1   202279   
1                2       28985                  2          1   202279   
2                2        9327                  3          0   202279   
3                2       45918                  4          1   202279   
4                2       30035                  5          0   202279   
...            ...         ...                ...        ...      ...   
32434484   3421083       39678                  6          1    25247   
32434485   3421083       11352                  7          0    25247   
32434486   3421083        4600                  8          0    25247   
32434487   3421083       24852                  9          1    25247   
32434488   3421083        5020                 10          1    25247   

          order_number  order_dow  order_hour_of_day  days_since_prior_order  \
0                    3